In [0]:
from pyspark.sql.functions import col,current_timestamp

source_name = "orders-item"
table_name = "order_items"

source_path = f"s3://reconciliation-platform/{source_name}/"
checkpoint_path = f"s3://reconciliation-platform/_checkpoints/{source_name}"
schema_path = f"s3://reconciliation-platform/_schemas/{source_name}"

df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format","csv")
    .option("cloudFiles.useManagedFileEvents","true")
    .option("cloudFiles.schemaLocation",schema_path)
    .option("cloudFiles.inferColumnTypes","true")
    .option("header","true")
    .load(source_path))

bronze_df = (df
    .withColumn("_source_file",col("_metadata.file_path"))
    .withColumn("_ingestion_timestamp",current_timestamp()))

query = (bronze_df.writeStream
    .format("delta")
    .option("checkpointLocation",checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(f"`reconciliation-platform`.bronze.{table_name}"))